In [ ]:
import numpy as np
from math import factorial

np.set_printoptions(suppress=True)

In [ ]:
class JackCarRentalEnv:
    def __init__(self, max_cars=20, max_move=5, rental_reward=10, move_cost=2,
                 rental_rate_1=3, rental_rate_2=2, return_rate_1=3, return_rate_2=2,
                 gamma=0.9):
        self.max_cars = max_cars
        self.max_move = max_move
        self.rental_reward = rental_reward
        self.move_cost = move_cost
        self.rental_rate_1 = rental_rate_1
        self.rental_rate_2 = rental_rate_2
        self.return_rate_1 = return_rate_1
        self.return_rate_2 = return_rate_2
        self.gamma = gamma
        self.actions = list(range(-self.max_move, self.max_move + 1))

    @staticmethod
    def poisson_prob(n, lam):
        if n < 0:
            return 0.0
        return np.exp(-lam) * (lam ** n) / factorial(n)

    def valid_action(self, state, action):
        i, j = state
        if action > 0:
            return i >= action
        if action < 0:
            return j >= -action
        return True

    def expected_return(self, state, action, V, gamma=None):
        if not self.valid_action(state, action):
            return -np.inf

        gamma = self.gamma if gamma is None else gamma
        i, j = state
        i_after = i - action
        j_after = j + action

        if i_after < 0 or j_after < 0:
            return -np.inf

        i_after = min(self.max_cars, i_after)
        j_after = min(self.max_cars, j_after)

        total = 0.0
        move_cost = self.move_cost * abs(action)

        for d1 in range(self.max_cars + 1):
            p_d1 = self.poisson_prob(d1, self.rental_rate_1)
            if p_d1 == 0.0:
                continue
            for d2 in range(self.max_cars + 1):
                p_d2 = self.poisson_prob(d2, self.rental_rate_2)
                if p_d2 == 0.0:
                    continue

                rented_1 = min(i_after, d1)
                rented_2 = min(j_after, d2)
                immediate_reward = self.rental_reward * (rented_1 + rented_2) - move_cost

                cars_1 = max(i_after - rented_1, 0)
                cars_2 = max(j_after - rented_2, 0)

                for r1 in range(self.max_cars + 1):
                    p_r1 = self.poisson_prob(r1, self.return_rate_1)
                    if p_r1 == 0.0:
                        continue
                    for r2 in range(self.max_cars + 1):
                        p_r2 = self.poisson_prob(r2, self.return_rate_2)
                        if p_r2 == 0.0:
                            continue

                        next_i = min(self.max_cars, cars_1 + r1)
                        next_j = min(self.max_cars, cars_2 + r2)
                        prob = p_d1 * p_d2 * p_r1 * p_r2
                        total += prob * (immediate_reward + gamma * V[next_i, next_j])

        return total


def policy_evaluation(env, policy, V=None, gamma=0.9, theta=1e-4):
    if V is None:
        V = np.zeros((env.max_cars + 1, env.max_cars + 1), dtype=np.float64)

    while True:
        delta = 0.0
        new_V = V.copy()

        for i in range(env.max_cars + 1):
            for j in range(env.max_cars + 1):
                action = int(policy[i, j])
                new_V[i, j] = env.expected_return((i, j), action, V, gamma)
                delta = max(delta, abs(new_V[i, j] - V[i, j]))

        if delta < theta:
            return new_V

        V = new_V


def policy_improvement(env, V, policy, gamma=0.9):
    policy_stable = True
    new_policy = policy.copy()

    for i in range(env.max_cars + 1):
        for j in range(env.max_cars + 1):
            state = (i, j)
            old_action = int(policy[i, j])
            best_action = old_action
            best_value = -np.inf

            for action in env.actions:
                if not env.valid_action(state, action):
                    continue
                q_value = env.expected_return(state, action, V, gamma)
                if q_value > best_value:
                    best_value = q_value
                    best_action = action

            new_policy[i, j] = best_action
            if old_action != best_action:
                policy_stable = False

    return policy_stable, new_policy


def policy_iteration(env, gamma=0.9, theta=1e-4):
    V = np.zeros((env.max_cars + 1, env.max_cars + 1), dtype=np.float64)
    policy = np.zeros((env.max_cars + 1, env.max_cars + 1), dtype=int)
    policy_stable = False
    iterations = 0

    while not policy_stable:
        V = policy_evaluation(env, policy, V, gamma, theta)
        policy_stable, policy = policy_improvement(env, V, policy, gamma)
        iterations += 1

    return policy, V


def print_policy(policy, rows=5, cols=5, title='Policy:'):
    print(title)
    for i in range(rows):
        print(i, policy[i, :cols])


def print_value_function(V, rows=5, cols=5, title='Value function:'):
    print(title)
    for i in range(rows):
        print(i, np.round(V[i, :cols], 3))

In [ ]:
env = JackCarRentalEnv(max_cars=20, gamma=0.9)
policy = np.zeros((env.max_cars + 1, env.max_cars + 1), dtype=int)
optimal_policy, optimal_V = policy_iteration(env, gamma=0.9, theta=1e-4)

print('Optimal policy slice (rows 0:5, cols 0:5):')
print_policy(optimal_policy, rows=5, cols=5)
print('\nOptimal value function slice (rows 0:5, cols 0:5):')
print_value_function(optimal_V, rows=5, cols=5)